🛠️ Configurazione Iniziale dei Dataset
Per prima cosa, definiamo i tre dataset forniti nelle prime due slide:

In [2]:
import pandas as pd
import numpy as np

# --- DATASET A: Vendite negozio ---
dati_a = {
    "venditore": ["Alice","Bob","Alice","Carlo","Bob","Alice","Carlo","Bob","Carlo","Alice","Bob","Carlo"],
    "regione": ["Nord","Sud","Nord","Centro","Sud","Nord","Centro","Sud","Centro","Nord","Sud","Centro"],
    "categoria": ["Elettronica","Abbigliamento","Elettronica","Alimentari","Elettronica","Abbigliamento","Elettronica","Alimentari","Abbigliamento","Alimentari","Elettronica","Abbigliamento"],
    "quantita": [5, 12, 3, 20, 8, 15, 6, 25, 10, 18, 4, 9],
    "prezzo": [200.0, 45.0, 200.0, 8.5, 200.0, 45.0, 200.0, 8.5, 45.0, 8.5, 200.0, 45.0],
    "data": ["2024-01-15","2024-01-22","2024-02-10","2024-01-08","2024-02-18","2024-03-05","2024-02-25","2024-03-12","2024-03-20","2024-01-30","2024-03-28","2024-02-14"]
}
df = pd.DataFrame(dati_a)
df['data'] = pd.to_datetime(df['data']) # Conversione esplicita in datetime
df['fatturato'] = df['quantita'] * df['prezzo'] # Calcolo preliminare del fatturato per i punti 1-4

# --- DATASET B: Serie storica temperature ---
giorni = pd.date_range(start="2024-01-01", periods=90, freq="D")
temperature = np.array([
    2.1, 1.8, 3.0, 0.5, -1.2, 2.4, 4.0, 3.5, 2.8, 1.0,
    0.2, -0.5, 1.5, 2.0, 3.8, 5.0, 4.2, 3.1, 2.7, 1.9,
    3.5, 4.8, 6.0, 5.5, 4.1, 3.0, 2.5, 4.0, 5.2, 6.8,
    7.0, 6.5, 5.8, 7.2, 8.0, 7.5, 6.9, 8.3, 9.0, 8.7,
    7.8, 9.2, 10.0, 9.5, 8.8, 10.3, 11.0, 10.6, 9.9, 11.2,
    12.0, 11.5, 10.8, 12.3, 13.0, 12.7, 11.9, 13.2, 14.0, 13.5,
    12.8, 14.2, 15.0, 14.6, 13.9, 15.3, 16.0, 15.5, 14.8, 16.2,
    17.0, 16.6, 15.9, 17.3, 18.0, 17.5, 16.8, 18.2, 19.0, 18.5,
    17.8, 19.2, 20.0, 19.5, 18.8, 20.3, 21.0, 20.5, 19.9, 21.2
])
ts = pd.Series(temperature, index=giorni, name="temperatura")

# --- DATASET C: Log accessi web ---
log_dati = {
    "timestamp": pd.to_datetime([
        "2024-03-01 08:15", "2024-03-01 09:30", "2024-03-01 09:45",
        "2024-03-02 10:00", "2024-03-02 11:20", "2024-03-03 08:05",
        "2024-03-03 14:30", "2024-03-04 09:00", "2024-03-04 16:45",
        "2024-03-05 08:30", "2024-03-05 12:00", "2024-03-05 17:30"
    ]),
    "utente": ["U1", "U2", "U1", "U3", "U2", "U1", "U3", "U2", "U1", "U3", "U1", "U2"],
    "pagina": ["home", "shop", "cart", "home", "shop", "blog", "home", "cart", "shop", "blog", "home", "cart"],
    "durata_s": [45, 120, 30, 60, 95, 80, 55, 40, 110, 75, 50, 35]
}
log = pd.DataFrame(log_dati)

💻 Risoluzione dei Quesiti (Slide 3)
1. GroupBy, aggregazione multipla

In [8]:
# Aggregazione con funzioni denominate per chiarezza di colonna
es1 = df.groupby('venditore').agg(
    fatturato_totale=('fatturato', 'sum'),
    quantita_media=('quantita', 'mean'),
    numero_transazioni=('venditore', 'count'),
    prezzo_massimo=('prezzo', 'max')
)
print(es1)

           fatturato_totale  quantita_media  numero_transazioni  \
venditore                                                         
Alice                2428.0           10.25                   4   
Bob                  3152.5           12.25                   4   
Carlo                2225.0           11.25                   4   

           prezzo_massimo  
venditore                  
Alice               200.0  
Bob                 200.0  
Carlo               200.0  


2. GroupBy, filter, soglia

In [6]:
# Filtra mantenendo solo i record dei venditori con media fatturato > 600€
es2 = df.groupby('venditore').filter(lambda x: x['fatturato'].mean() > 600)
print(es2)

   venditore regione      categoria  quantita  prezzo       data  fatturato
0      Alice    Nord    Elettronica         5   200.0 2024-01-15     1000.0
1        Bob     Sud  Abbigliamento        12    45.0 2024-01-22      540.0
2      Alice    Nord    Elettronica         3   200.0 2024-02-10      600.0
4        Bob     Sud    Elettronica         8   200.0 2024-02-18     1600.0
5      Alice    Nord  Abbigliamento        15    45.0 2024-03-05      675.0
7        Bob     Sud     Alimentari        25     8.5 2024-03-12      212.5
9      Alice    Nord     Alimentari        18     8.5 2024-01-30      153.0
10       Bob     Sud    Elettronica         4   200.0 2024-03-28      800.0


3. Pivot (Fatturato per venditore e mese)

In [16]:
# 1. Creiamo una colonna temporanea o definitiva per il mese (es. 'mese')
df['mese'] = df['data'].dt.month

# 2. Generiamo la pivot table usando il nome della colonna appena creata
es3 = df.pivot_table(
    index='venditore', 
    columns='mese',      # Passiamo la stringa, non la Series!
    values='fatturato', 
    aggfunc='sum', 
    margins=True         # Ora i totali funzioneranno perfettamente
)

print(es3)

mese            1       2       3     All
venditore                                
Alice      1153.0   600.0   675.0  2428.0
Bob         540.0  1600.0  1012.5  3152.5
Carlo       170.0  1605.0   450.0  2225.0
All        1863.0  3805.0  2137.5  7805.5


4. Pivot Multi-livello (Regione e Categoria)

In [9]:
es4 = df.pivot_table(
    index='regione', 
    columns='categoria', 
    values='fatturato', 
    aggfunc=['sum', 'mean']
)
print(es4)

                    sum                                 mean             \
categoria Abbigliamento Alimentari Elettronica Abbigliamento Alimentari   
regione                                                                   
Centro            855.0      170.0      1200.0         427.5      170.0   
Nord              675.0      153.0      1600.0         675.0      153.0   
Sud               540.0      212.5      2400.0         540.0      212.5   

                       
categoria Elettronica  
regione                
Centro         1200.0  
Nord            800.0  
Sud            1200.0  


5. Conversione e Parsing delle date

In [10]:
# Estrazione di informazioni strutturate per il parsing
df['anno'] = df['data'].dt.year
df['mese'] = df['data'].dt.month
df['giorno'] = df['data'].dt.day
print(df[['data', 'anno', 'mese', 'giorno']].head())

        data  anno  mese  giorno
0 2024-01-15  2024     1      15
1 2024-01-22  2024     1      22
2 2024-02-10  2024     2      10
3 2024-01-08  2024     1       8
4 2024-02-18  2024     2      18


6. Estrazione di componenti temporali e analisi distribuzioni

In [11]:
df['giorno_settimana'] = df['data'].dt.day_name() # Nome del giorno in inglese (o dt.weekday per numero)
df['settimana_anno'] = df['data'].dt.isocalendar().week

# Distribuzione del fatturato per giorno della settimana
distribuzione_giornaliera = df.groupby('giorno_settimana')['fatturato'].sum()
print(distribuzione_giornaliera)

giorno_settimana
Monday       1710.0
Saturday      600.0
Sunday       2800.0
Thursday      800.0
Tuesday      1040.5
Wednesday     855.0
Name: fatturato, dtype: float64


7. Serie Storiche – GroupBy su DatetimeIndex

In [12]:
# Calcolo statistiche descrittive complete mensili (Gennaio, Febbraio, Marzo)
statistiche_mensili = ts.groupby(ts.index.month).describe()
print(statistiche_mensili)

   count       mean       std   min    25%   50%    75%   max
1   31.0   3.103226  1.988883  -1.2   1.95   3.0   4.15   7.0
2   29.0  10.037931  2.265961   5.8   8.30  10.0  11.90  14.0
3   30.0  17.376667  2.288002  12.8  15.60  17.4  19.15  21.2


8. Colonne Calcolate e Formattazione Finale

In [15]:
# Ripristiniamo/aggiorniamo le colonne richieste in sequenza logica
df['fatturato'] = df['quantita'] * df['prezzo']
df['iva'] = df['fatturato'] * 0.22
df['fatturato_ivato'] = df['fatturato'] + df['iva']

# Condizioni logiche per lo 'sconto_pct'
condizioni = [
    df['categoria'] == 'Elettronica',
    df['categoria'] == 'Abbigliamento'
]
scelte = [0.15, 0.08]
df['sconto_pct'] = np.select(condizioni, choicelist=scelte, default=0.05)

# Calcolo del netto finale
df['netto_finale'] = df['fatturato_ivato'] * (1 - df['sconto_pct'])

# Arrotondamento generale a 2 decimali
df = df.round(2)

# Stampa selettiva delle colonne richieste
risultato_finale = df[['venditore', 'categoria', 'fatturato', 'iva', 'fatturato_ivato', 'sconto_pct', 'netto_finale']]
print(risultato_finale)

   venditore      categoria  fatturato     iva  fatturato_ivato  sconto_pct  \
0      Alice    Elettronica     1000.0  220.00          1220.00        0.15   
1        Bob  Abbigliamento      540.0  118.80           658.80        0.08   
2      Alice    Elettronica      600.0  132.00           732.00        0.15   
3      Carlo     Alimentari      170.0   37.40           207.40        0.05   
4        Bob    Elettronica     1600.0  352.00          1952.00        0.15   
5      Alice  Abbigliamento      675.0  148.50           823.50        0.08   
6      Carlo    Elettronica     1200.0  264.00          1464.00        0.15   
7        Bob     Alimentari      212.5   46.75           259.25        0.05   
8      Carlo  Abbigliamento      450.0   99.00           549.00        0.08   
9      Alice     Alimentari      153.0   33.66           186.66        0.05   
10       Bob    Elettronica      800.0  176.00           976.00        0.15   
11     Carlo  Abbigliamento      405.0   89.10      

C:\Users\A1627apulia\AppData\Local\Temp\ipykernel_29636\731805579.py:18: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  df = df.round(2)
